In [27]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\user\AppData\Local\Temp\ipykernel_8056\3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [28]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [29]:
## READ DATA FOR THE FRAUDULENT EMAIL KAGGLE CHALLENGE
data = pd.read_csv("../data/kg_train.csv", encoding="latin-1")

# REDUCE THE TRAINING SET TO SPEED UP DEVELOPMENT.
# MODIFY FOR FINAL SYSTEM

#data = data.head(1000)
print(data.shape)
data.fillna("", inplace=True)


(5964, 2)


In [30]:
data.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


### Let's divide the training and test set into two partitions

In [31]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data, test_size=0.2, random_state=42, shuffle=True)

print(data_train.shape, data_val.shape)


(4771, 2) (1193, 2)


## Data Preprocessing

In [32]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer("english")


!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [33]:
import re

def clean_html(text) :
    # REMOVE INLINE JS/CSS
    text = re.sub(r"<script.*?>.*?</script>", "", text, flags=re.DOTALL)
    text = re.sub(r"<style.*?>.*?</style>", "", text, flags=re.DOTALL)

    # REMOVE HTML COMMENTS
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    
    # REMOVE REMAINING HTML TAGS
    text = re.sub(r"<.*?>", "", text)
    
    return text

data_train["clean_text"] = data_train["text"].apply(clean_html)
data_val["clean_text"] = data_val["text"].apply(clean_html)



In [34]:
data_train

,text,label,clean_text
3821,back -- we need to talkCall only my berry.,0,back -- we need to talkCall only my berry.
2407,What is mullen's first name?,0,What is mullen's first name?
1173,Greetings from Dubai=2CThis letter must come t...,1,Greetings from Dubai=2CThis letter must come t...
4231,Sullivan Jacob J <SullivanJJ@state.gov>Friday ...,0,Sullivan Jacob J Friday December 4 2009 3:31 A...
1666,Fyi,0,Fyi
...,...,...,...
3772,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",1,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE...."
5191,YesThe Argentina desk is working this issue on...,0,YesThe Argentina desk is working this issue on...
5226,FSMAIL.NET</A> All payments offshore has been ...,1,FSMAIL.NET All payments offshore has been reca...
5390,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",1,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ..."


In [35]:
data_val

,text,label,clean_text
5713,MS -- Ijust spoke to Maggie and she can meet w...,0,MS -- Ijust spoke to Maggie and she can meet w...
810,RELEASE IN PART B6Ok,0,RELEASE IN PART B6Ok
2488,Yes and we can certainly do,0,Yes and we can certainly do
2960,Looks like there will be a three-way meeting i...,0,Looks like there will be a three-way meeting i...
4680,Remain blessed in the name of the Lord.Yours i...,1,Remain blessed in the name of the Lord.Yours i...
...,...,...,...
5214,To serve as a guardian of this fund since I ...,1,To serve as a guardian of this fund since I ...
925,When is Steinberg call?,0,When is Steinberg call?
2985,My flight takes off at 8 so call before if you...,0,My flight takes off at 8 so call before if you...
1886,"My name is John paul, a member of Independent ...",1,"My name is John paul, a member of Independent ..."


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [40]:
def preprocess_text(text) :
    text = re.sub(r"\W", " ", text)                  # REMOVE SPECIAL CHARS
    text = re.sub(r"\d+", " ", text)                 # REMOVE NUMBERS
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text )     # REMOVE SINGLE CHARS
    text = re.sub(r"^[a-zA-Z]\s+", " ", text)        # REMOVE SINGLE CHAR AT START
    text = re.sub(r"\s+", " ", text)                # Multiple spaces → one
    text = text.lower()                             # LOWERCASE
    return text

data_train["preprocessed_text"] = data_train["clean_text"].apply(preprocess_text)
data_val["preprocessed_text"] = data_val["clean_text"].apply(preprocess_text)


In [41]:
data_train

,text,label,clean_text,preprocessed_text
3821,back -- we need to talkCall only my berry.,0,back -- we need to talkCall only my berry.,back we need to talkcall only my berry
2407,What is mullen's first name?,0,What is mullen's first name?,what is mullen first name
1173,Greetings from Dubai=2CThis letter must come t...,1,Greetings from Dubai=2CThis letter must come t...,greetings from dubai cthis letter must come to...
4231,Sullivan Jacob J <SullivanJJ@state.gov>Friday ...,0,Sullivan Jacob J Friday December 4 2009 3:31 A...,sullivan jacob friday december amiranthe eu me...
1666,Fyi,0,Fyi,fyi
...,...,...,...,...
3772,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",1,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",dear friend seeking your immediate assistance ...
5191,YesThe Argentina desk is working this issue on...,0,YesThe Argentina desk is working this issue on...,yesthe argentina desk is working this issue on...
5226,FSMAIL.NET</A> All payments offshore has been ...,1,FSMAIL.NET All payments offshore has been reca...,fsmail net all payments offshore has been reca...
5390,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",1,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",dear friend re swift telegraphic transfer of u...


In [42]:
data_val

,text,label,clean_text,preprocessed_text
5713,MS -- Ijust spoke to Maggie and she can meet w...,0,MS -- Ijust spoke to Maggie and she can meet w...,ms ijust spoke to maggie and she can meet with...
810,RELEASE IN PART B6Ok,0,RELEASE IN PART B6Ok,release in part ok
2488,Yes and we can certainly do,0,Yes and we can certainly do,yes and we can certainly do
2960,Looks like there will be a three-way meeting i...,0,Looks like there will be a three-way meeting i...,looks like there will be three way meeting in ...
4680,Remain blessed in the name of the Lord.Yours i...,1,Remain blessed in the name of the Lord.Yours i...,remain blessed in the name of the lord yours i...
...,...,...,...,...
5214,To serve as a guardian of this fund since I ...,1,To serve as a guardian of this fund since I ...,to serve as guardian of this fund since am yo...
925,When is Steinberg call?,0,When is Steinberg call?,when is steinberg call
2985,My flight takes off at 8 so call before if you...,0,My flight takes off at 8 so call before if you...,my flight takes off at so call before if you h...
1886,"My name is John paul, a member of Independent ...",1,"My name is John paul, a member of Independent ...",my name is john paul member of independent com...


## Now let's work on removing stopwords
Remove the stopwords.

In [43]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text) :
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

data_train["preprocessed_text_stopwords"] = data_train["preprocessed_text"].apply(remove_stopwords)
data_val["preprocessed_text_stopwords"] = data_val["preprocessed_text"].apply(remove_stopwords)


In [44]:
data_train

,text,label,clean_text,preprocessed_text,preprocessed_text_stopwords
3821,back -- we need to talkCall only my berry.,0,back -- we need to talkCall only my berry.,back we need to talkcall only my berry,back need talkcall berry
2407,What is mullen's first name?,0,What is mullen's first name?,what is mullen first name,mullen first name
1173,Greetings from Dubai=2CThis letter must come t...,1,Greetings from Dubai=2CThis letter must come t...,greetings from dubai cthis letter must come to...,greetings dubai cthis letter must come big sur...
4231,Sullivan Jacob J <SullivanJJ@state.gov>Friday ...,0,Sullivan Jacob J Friday December 4 2009 3:31 A...,sullivan jacob friday december amiranthe eu me...,sullivan jacob friday december amiranthe eu me...
1666,Fyi,0,Fyi,fyi,fyi
...,...,...,...,...,...
3772,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",1,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",dear friend seeking your immediate assistance ...,dear friend seeking immediate assistance pleas...
5191,YesThe Argentina desk is working this issue on...,0,YesThe Argentina desk is working this issue on...,yesthe argentina desk is working this issue on...,yesthe argentina desk working issue behalf tou...
5226,FSMAIL.NET</A> All payments offshore has been ...,1,FSMAIL.NET All payments offshore has been reca...,fsmail net all payments offshore has been reca...,fsmail net payments offshore recalled back ape...
5390,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",1,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",dear friend re swift telegraphic transfer of u...,dear friend swift telegraphic transfer us elev...


In [45]:
data_val

,text,label,clean_text,preprocessed_text,preprocessed_text_stopwords
5713,MS -- Ijust spoke to Maggie and she can meet w...,0,MS -- Ijust spoke to Maggie and she can meet w...,ms ijust spoke to maggie and she can meet with...,ms ijust spoke maggie meet dinner tuesday nigh...
810,RELEASE IN PART B6Ok,0,RELEASE IN PART B6Ok,release in part ok,release part ok
2488,Yes and we can certainly do,0,Yes and we can certainly do,yes and we can certainly do,yes certainly
2960,Looks like there will be a three-way meeting i...,0,Looks like there will be a three-way meeting i...,looks like there will be three way meeting in ...,looks like three way meeting turkey tomorrow
4680,Remain blessed in the name of the Lord.Yours i...,1,Remain blessed in the name of the Lord.Yours i...,remain blessed in the name of the lord yours i...,remain blessed name lord christ mr harry hilal
...,...,...,...,...,...
5214,To serve as a guardian of this fund since I ...,1,To serve as a guardian of this fund since I ...,to serve as guardian of this fund since am yo...,serve guardian fund since young boy years with...
925,When is Steinberg call?,0,When is Steinberg call?,when is steinberg call,steinberg call
2985,My flight takes off at 8 so call before if you...,0,My flight takes off at 8 so call before if you...,my flight takes off at so call before if you h...,flight takes call minute cdm
1886,"My name is John paul, a member of Independent ...",1,"My name is John paul, a member of Independent ...",my name is john paul member of independent com...,name john paul member independent committee em...


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [46]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text) :
    return " ".join([lemmatizer.lemmatize(w) for w in text.split()])

data_train["preprocessed_text_lemmatized"] = data_train["preprocessed_text_stopwords"].apply(lemmatize_text)
data_val["preprocessed_text_lemmatized"] = data_val["preprocessed_text_stopwords"].apply(lemmatize_text)


In [47]:
data_train

,text,label,clean_text,preprocessed_text,preprocessed_text_stopwords,preprocessed_text_lemmatized
3821,back -- we need to talkCall only my berry.,0,back -- we need to talkCall only my berry.,back we need to talkcall only my berry,back need talkcall berry,back need talkcall berry
2407,What is mullen's first name?,0,What is mullen's first name?,what is mullen first name,mullen first name,mullen first name
1173,Greetings from Dubai=2CThis letter must come t...,1,Greetings from Dubai=2CThis letter must come t...,greetings from dubai cthis letter must come to...,greetings dubai cthis letter must come big sur...,greeting dubai cthis letter must come big surp...
4231,Sullivan Jacob J <SullivanJJ@state.gov>Friday ...,0,Sullivan Jacob J Friday December 4 2009 3:31 A...,sullivan jacob friday december amiranthe eu me...,sullivan jacob friday december amiranthe eu me...,sullivan jacob friday december amiranthe eu me...
1666,Fyi,0,Fyi,fyi,fyi,fyi
...,...,...,...,...,...,...
3772,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",1,"Dear Friend,SEEKING YOUR IMMEDIATE ASSISTANCE....",dear friend seeking your immediate assistance ...,dear friend seeking immediate assistance pleas...,dear friend seeking immediate assistance pleas...
5191,YesThe Argentina desk is working this issue on...,0,YesThe Argentina desk is working this issue on...,yesthe argentina desk is working this issue on...,yesthe argentina desk working issue behalf tou...,yesthe argentina desk working issue behalf tou...
5226,FSMAIL.NET</A> All payments offshore has been ...,1,FSMAIL.NET All payments offshore has been reca...,fsmail net all payments offshore has been reca...,fsmail net payments offshore recalled back ape...,fsmail net payment offshore recalled back apex...
5390,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",1,"Dear Friend,RE; SWIFT/TELEGRAPHIC TRANSFER OF ...",dear friend re swift telegraphic transfer of u...,dear friend swift telegraphic transfer us elev...,dear friend swift telegraphic transfer u eleve...


In [48]:
data_val

,text,label,clean_text,preprocessed_text,preprocessed_text_stopwords,preprocessed_text_lemmatized
5713,MS -- Ijust spoke to Maggie and she can meet w...,0,MS -- Ijust spoke to Maggie and she can meet w...,ms ijust spoke to maggie and she can meet with...,ms ijust spoke maggie meet dinner tuesday nigh...,m ijust spoke maggie meet dinner tuesday night...
810,RELEASE IN PART B6Ok,0,RELEASE IN PART B6Ok,release in part ok,release part ok,release part ok
2488,Yes and we can certainly do,0,Yes and we can certainly do,yes and we can certainly do,yes certainly,yes certainly
2960,Looks like there will be a three-way meeting i...,0,Looks like there will be a three-way meeting i...,looks like there will be three way meeting in ...,looks like three way meeting turkey tomorrow,look like three way meeting turkey tomorrow
4680,Remain blessed in the name of the Lord.Yours i...,1,Remain blessed in the name of the Lord.Yours i...,remain blessed in the name of the lord yours i...,remain blessed name lord christ mr harry hilal,remain blessed name lord christ mr harry hilal
...,...,...,...,...,...,...
5214,To serve as a guardian of this fund since I ...,1,To serve as a guardian of this fund since I ...,to serve as guardian of this fund since am yo...,serve guardian fund since young boy years with...,serve guardian fund since young boy year witho...
925,When is Steinberg call?,0,When is Steinberg call?,when is steinberg call,steinberg call,steinberg call
2985,My flight takes off at 8 so call before if you...,0,My flight takes off at 8 so call before if you...,my flight takes off at so call before if you h...,flight takes call minute cdm,flight take call minute cdm
1886,"My name is John paul, a member of Independent ...",1,"My name is John paul, a member of Independent ...",my name is john paul member of independent com...,name john paul member independent committee em...,name john paul member independent committee em...


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [49]:
from collections import Counter

ham_words = Counter()
spam_words = Counter()

for _, row in data_train.iterrows() :
    words = row["preprocessed_text_lemmatized"].split()
    if row["label"] == "ham" :
        ham_words.update(words)
    else :
        spam_words.update(words)

print("TOP HAM WORDS :", ham_words.most_common(10))
print("TOP SPAM WORDS :", spam_words.most_common(10))


TOP HAM WORDS : []
TOP SPAM WORDS : [('money', 4724), ('account', 4034), ('bank', 3931), ('u', 3703), ('fund', 3487), ('e', 3174), ('nbsp', 2980), ('â', 2762), ('mr', 2599), ('country', 2482)]


## Extra features

In [50]:
# WE ADD TO THE ORIGINAL DATFRAME TWO ADDITIONAL INDICATORS (MONEY SYMBOLS AND SUSPICIOUS WORDS).
money_simbol_list = "|".join(["euro", "dollar", "pound", "€", r"\$"])
suspicious_words = "|".join(["free", "cheap", "sex", "money", "account", "bank", "fund", "transfer", "transaction", "win", "deposit", "password"])

data_train["money_mark"] = data_train["preprocessed_text_lemmatized"].str.contains(money_simbol_list)*1
data_train["suspicious_words"] = data_train["preprocessed_text_lemmatized"].str.contains(suspicious_words)*1
data_train["text_len"] = data_train["preprocessed_text_lemmatized"].apply(lambda x : len(x))

data_val["money_mark"] = data_val["preprocessed_text_lemmatized"].str.contains(money_simbol_list)*1
data_val["suspicious_words"] = data_val["preprocessed_text_lemmatized"].str.contains(suspicious_words)*1
data_val["text_len"] = data_val["preprocessed_text_lemmatized"].apply(lambda x : len(x))

data_train.head()


,text,label,clean_text,preprocessed_text,preprocessed_text_stopwords,preprocessed_text_lemmatized,money_mark,suspicious_words,text_len
3821,back -- we need to talkCall only my berry.,0,back -- we need to talkCall only my berry.,back we need to talkcall only my berry,back need talkcall berry,back need talkcall berry,0,0,24
2407,What is mullen's first name?,0,What is mullen's first name?,what is mullen first name,mullen first name,mullen first name,0,0,17
1173,Greetings from Dubai=2CThis letter must come t...,1,Greetings from Dubai=2CThis letter must come t...,greetings from dubai cthis letter must come to...,greetings dubai cthis letter must come big sur...,greeting dubai cthis letter must come big surp...,1,1,2435
4231,Sullivan Jacob J <SullivanJJ@state.gov>Friday ...,0,Sullivan Jacob J Friday December 4 2009 3:31 A...,sullivan jacob friday december amiranthe eu me...,sullivan jacob friday december amiranthe eu me...,sullivan jacob friday december amiranthe eu me...,0,1,417
1666,Fyi,0,Fyi,fyi,fyi,fyi,0,0,3


## How would work the Bag of Words with Count Vectorizer concept?

In [51]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X_train_cv = cv.fit_transform(data_train["preprocessed_text_lemmatized"])
X_val_cv = cv.transform(data_val["preprocessed_text_lemmatized"])

print(X_train_cv.shape, X_val_cv.shape)


(4771, 85081) (1193, 85081)


In [53]:
X_train_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 422040 stored elements and shape (4771, 85081)>

In [54]:
X_val_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 84292 stored elements and shape (1193, 85081)>

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [52]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text_lemmatized"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text_lemmatized"])

print(X_train_tfidf.shape, X_val_tfidf.shape)


(4771, 85081) (1193, 85081)


In [55]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 422040 stored elements and shape (4771, 85081)>

In [56]:
X_val_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 84292 stored elements and shape (1193, 85081)>

## And the Train a Classifier?

In [57]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

clf = MultinomialNB()
clf.fit(X_train_tfidf, data_train["label"])

preds = clf.predict(X_val_tfidf)
print(classification_report(data_val["label"], preds))


              precision    recall  f1-score   support

           0       1.00      0.94      0.97       674
           1       0.92      0.99      0.96       519

    accuracy                           0.96      1193
   macro avg       0.96      0.97      0.96      1193
weighted avg       0.96      0.96      0.96      1193



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [59]:
from sklearn.metrics import accuracy_score
import numpy as np

def evaluate_model(X_train, X_val, extra_train=None, extra_val=None) :
    if extra_train is not None :
        X_train = np.hstack([X_train.toarray(), extra_train])
        X_val = np.hstack([X_val.toarray(), extra_val])
    
    clf = MultinomialNB()
    clf.fit(X_train, data_train["label"])
    preds = clf.predict(X_val)
    return accuracy_score(data_val["label"], preds)


# EXTRA FEATURES
extra_train = data_train[["money_mark", "suspicious_words", "text_len"]].values
extra_val = data_val[["money_mark", "suspicious_words", "text_len"]].values

print("BOW ONLY :", evaluate_model(X_train_cv, X_val_cv))
print("TF-IDF ONLY :", evaluate_model(X_train_tfidf, X_val_tfidf))
print("BOW + EXTRA :", evaluate_model(X_train_cv, X_val_cv, extra_train, extra_val))
print("TF-IDF + EXTRA :", evaluate_model(X_train_tfidf, X_val_tfidf, extra_train, extra_val))


BOW ONLY : 0.9681475272422464
TF-IDF ONLY : 0.9622799664710813


MemoryError: Unable to allocate 3.02 GiB for an array with shape (4771, 85084) and data type int64